# ONH Skeletonisation Strategy Comparison

This notebook generates a report-ready comparison figure for ONH skeletonisation. The three columns are named by qualitative algorithm strategy rather than by version number.

The binary input is generated using the selected ONH binarisation method. The skeletonisation strategies differ only in pruning strength:

- **Balanced pruning**: moderate pruning, selected as the most suitable default for the ONH pipeline.
- **Fine-vessel preserving pruning**: lighter pruning, keeps more short fine-vessel branches.
- **Aggressive graph-cleaning pruning**: stronger pruning, removes more short spurs before graph analysis.

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from scipy.ndimage import convolve, distance_transform_edt, binary_dilation
from skimage.morphology import medial_axis, remove_small_objects
from skimage.measure import label

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "gui" / "build" / "modules" / "onh_binarise_image.py").exists():
            if (candidate / "algorithms test images").exists():
                return candidate
    raise FileNotFoundError("Could not find the OCTAVA project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
TEST_DIR = PROJECT_ROOT / "algorithms test images"
GUI_BUILD_DIR = PROJECT_ROOT / "gui" / "build"
MODULE_DIR = GUI_BUILD_DIR / "modules"

for item in (str(GUI_BUILD_DIR), str(MODULE_DIR)):
    if item not in sys.path:
        sys.path.insert(0, item)

from modules.onh_binarise_image import binarise_onh_image

METADATA_PATH = TEST_DIR / "test images information.xlsx"
OUTPUT_DIR = TEST_DIR / "notebook_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Test image folder:", TEST_DIR)
print("Metadata:", METADATA_PATH)

## Load Test Images

The annular ROI is fixed at 2-4 mm diameter within a 4.5 mm x 4.5 mm ONH scan.

In [ ]:
IMAGE_SIZE_PX = 1500
SCAN_SIZE_MM = 4.5
INNER_DIAMETER_MM = 2.0
OUTER_DIAMETER_MM = 4.0

metadata = pd.read_excel(METADATA_PATH)
metadata

In [ ]:
def read_grayscale_image(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), dtype=np.float64)


def resize_to_scan(image: np.ndarray, image_size_px: int = IMAGE_SIZE_PX) -> np.ndarray:
    interpolation = cv2.INTER_AREA if max(image.shape) > image_size_px else cv2.INTER_CUBIC
    return cv2.resize(image, (image_size_px, image_size_px), interpolation=interpolation).astype(np.float64)


def annulus_mask(
    image_shape: tuple[int, int],
    center_x: float,
    center_y: float,
    inner_diameter_mm: float = INNER_DIAMETER_MM,
    outer_diameter_mm: float = OUTER_DIAMETER_MM,
    scan_size_mm: float = SCAN_SIZE_MM,
) -> np.ndarray:
    h, w = image_shape
    px_per_mm = w / scan_size_mm
    inner_radius_px = (inner_diameter_mm / 2.0) * px_per_mm
    outer_radius_px = (outer_diameter_mm / 2.0) * px_per_mm
    yy, xx = np.ogrid[:h, :w]
    distance = np.sqrt((xx - center_x) ** 2 + (yy - center_y) ** 2)
    return (distance >= inner_radius_px) & (distance <= outer_radius_px)


def load_cases(metadata: pd.DataFrame) -> list[dict]:
    cases = []
    for _, row in metadata.iterrows():
        image_path = TEST_DIR / str(row["filename"])
        raw = read_grayscale_image(image_path)
        image = resize_to_scan(raw)
        center_x = float(row["center_x"])
        center_y = float(row["center_y"])
        roi_mask = annulus_mask(image.shape, center_x, center_y)
        device = str(row["device"]).strip()
        quality = str(row["quality"]).strip()
        cases.append(
            {
                "name": f"{device} {quality}",
                "filename": str(row["filename"]),
                "path": image_path,
                "raw_shape": raw.shape,
                "image": image,
                "roi_mask": roi_mask,
                "center_x": center_x,
                "center_y": center_y,
                "device": device,
                "quality": quality,
            }
        )
    return cases


cases = load_cases(metadata)
[(case["name"], case["raw_shape"], case["image"].shape, (case["center_x"], case["center_y"])) for case in cases]

## Generate Binary Masks

The same binary mask is used for all three skeletonisation strategies, so the comparison reflects pruning behaviour only.

In [ ]:
class NotebookVar:
    def __init__(self, value):
        self.value = value
    def get(self):
        return self.value


class NotebookCanvas:
    status_id = 1
    frangi_bool = NotebookVar(1)
    frangi_min = NotebookVar(1)
    frangi_max = NotebookVar(4)
    med_filt_bool = NotebookVar(0)
    kernel_size = NotebookVar(3)

    def itemconfigure(self, *args, **kwargs):
        pass

    def update_idletasks(self):
        pass


def generate_binary(case: dict) -> np.ndarray:
    canvas = NotebookCanvas()
    canvas.onh_mask_img = case["image"].astype(np.float64)
    canvas.onh_mask_arr = case["roi_mask"].astype(bool)
    binarise_onh_image(canvas, display=False)
    return canvas.binary_image.astype(np.uint8)


binary_results = {case["name"]: generate_binary(case) for case in cases}
print("Generated binary masks using the selected ONH binarisation method.")

## Skeletonisation Strategies

The selected strategy is **Balanced pruning**, because it removes small artefacts while preserving more plausible fine-vessel continuity than the aggressive pruning strategy.

In [ ]:
def neighbour_count_8(skeleton: np.ndarray) -> np.ndarray:
    kernel = np.array(
        [[1, 1, 1],
         [1, 0, 1],
         [1, 1, 1]],
        dtype=np.uint8,
    )
    return convolve(skeleton.astype(np.uint8), kernel, mode="constant", cval=0)


def endpoint_and_branchpoint_masks(skeleton: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    skel = skeleton.astype(bool)
    n = neighbour_count_8(skel)
    endpoints = skel & (n == 1)
    branchpoints = skel & (n >= 3)
    return endpoints, branchpoints


def skeleton_component_count(skeleton: np.ndarray) -> int:
    _, n_components = label(skeleton.astype(bool), connectivity=2, return_num=True)
    return int(n_components)


def prune_short_spurs(skeleton: np.ndarray, min_branch_length: int = 12, max_iterations: int = 10) -> np.ndarray:
    skel = skeleton.astype(bool).copy()

    for _ in range(max_iterations):
        neighbour_count = neighbour_count_8(skel)
        endpoints = np.argwhere(skel & (neighbour_count == 1))
        if len(endpoints) == 0:
            break

        pixels_to_remove = []
        for endpoint in endpoints:
            path = []
            current = tuple(endpoint)
            previous = None

            for _step in range(min_branch_length + 1):
                path.append(current)
                y, x = current
                neighbours = []

                for dy in (-1, 0, 1):
                    for dx in (-1, 0, 1):
                        if dy == 0 and dx == 0:
                            continue
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < skel.shape[0] and 0 <= nx < skel.shape[1] and skel[ny, nx]:
                            if previous is None or (ny, nx) != previous:
                                neighbours.append((ny, nx))

                if len(neighbours) == 0:
                    break

                if len(neighbours) >= 2:
                    if len(path) <= min_branch_length:
                        pixels_to_remove.extend(path[:-1])
                    break

                previous = current
                current = neighbours[0]

        if not pixels_to_remove:
            break

        for y, x in pixels_to_remove:
            skel[y, x] = False

    return skel


def run_skeletonisation_strategy(binary: np.ndarray, roi_mask: np.ndarray, params: dict) -> dict:
    roi_mask = roi_mask.astype(bool)
    vessel_mask = (binary > 0) & roi_mask
    image_size = vessel_mask.shape[0]

    pre_min_area = max(
        int(params.get("pre_min_area_min", 20)),
        int(image_size * image_size * float(params.get("pre_min_area_scale", 0.00001))),
    )
    vessel_mask_clean = remove_small_objects(vessel_mask, min_size=pre_min_area, connectivity=2)
    vessel_mask_clean = vessel_mask_clean & roi_mask

    raw_skeleton, medial_distance = medial_axis(vessel_mask_clean, return_distance=True)
    raw_skeleton = raw_skeleton & roi_mask
    distance_map = distance_transform_edt(vessel_mask_clean)

    skeleton_min_size = max(1, int(params.get("skeleton_min_size", 8)))
    component_pruned = remove_small_objects(raw_skeleton, min_size=skeleton_min_size, connectivity=2)
    component_pruned = component_pruned & roi_mask

    spur_length = int(params.get("spur_prune_length", 12))
    max_iterations = int(params.get("spur_prune_iterations", 10))
    skeleton = prune_short_spurs(component_pruned, min_branch_length=spur_length, max_iterations=max_iterations)
    skeleton = skeleton & roi_mask

    endpoints, branchpoints = endpoint_and_branchpoint_masks(skeleton)

    return {
        "vessel_mask_clean": vessel_mask_clean,
        "raw_skeleton": raw_skeleton,
        "component_pruned": component_pruned,
        "skeleton": skeleton,
        "distance_map": distance_map,
        "endpoints": endpoints,
        "branchpoints": branchpoints,
        "pre_min_area": pre_min_area,
        "skeleton_min_size": skeleton_min_size,
        "spur_prune_length": spur_length,
        "params": params.copy(),
    }

In [ ]:
SKELETON_STRATEGIES = {
    "Balanced pruning": {
        "pre_min_area_scale": 0.00001,
        "pre_min_area_min": 20,
        "skeleton_min_size": 8,
        "spur_prune_length": 15,
        "spur_prune_iterations": 10,
    },
    "Fine-vessel preserving pruning": {
        "pre_min_area_scale": 0.000005,
        "pre_min_area_min": 10,
        "skeleton_min_size": 5,
        "spur_prune_length": 8,
        "spur_prune_iterations": 6,
    },
    "Aggressive graph-cleaning pruning": {
        "pre_min_area_scale": 0.00002,
        "pre_min_area_min": 30,
        "skeleton_min_size": 12,
        "spur_prune_length": 22,
        "spur_prune_iterations": 12,
    },
}

skeleton_results = {}
for strategy_name, params in SKELETON_STRATEGIES.items():
    skeleton_results[strategy_name] = {}
    for case in cases:
        skeleton_results[strategy_name][case["name"]] = run_skeletonisation_strategy(
            binary_results[case["name"]],
            case["roi_mask"],
            params,
        )

print("Skeletonisation strategies complete.")

## Report-Ready Figure

This is the main figure intended for the report.

In [ ]:
def skeleton_display_image(skeleton: np.ndarray, dilation_iter: int = 1) -> np.ndarray:
    return binary_dilation(skeleton.astype(bool), iterations=dilation_iter).astype(np.uint8) * 255


def plot_report_skeleton_comparison(cases: list[dict], skeleton_results: dict, save_path: Path | None = None) -> None:
    strategy_names = list(skeleton_results.keys())
    fig, axes = plt.subplots(
        len(cases),
        len(strategy_names),
        figsize=(4.2 * len(strategy_names), 3.9 * len(cases)),
        squeeze=False,
    )

    for r, case in enumerate(cases):
        for c, strategy_name in enumerate(strategy_names):
            ax = axes[r, c]
            skeleton = skeleton_results[strategy_name][case["name"]]["skeleton"]
            ax.imshow(skeleton_display_image(skeleton, dilation_iter=1), cmap="gray", vmin=0, vmax=255)
            if r == 0:
                ax.set_title(strategy_name)
            if c == 0:
                ax.set_ylabel(case["name"], rotation=90, labelpad=8)
            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle("ONH skeletonisation strategy comparison", y=0.995)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print("Saved report-ready figure to:", save_path)

    plt.show()


REPORT_FIGURE_PATH = OUTPUT_DIR / "onh_skeletonisation_strategy_comparison.png"
plot_report_skeleton_comparison(cases, skeleton_results, save_path=REPORT_FIGURE_PATH)

Suggested figure caption:

**Figure X.** Comparison of three ONH skeletonisation pruning strategies using the same binarised ONH images. Balanced pruning was selected for the ONH pipeline because it reduced small artefacts while preserving the main radial vessel network and fine-vessel continuity better than the aggressive graph-cleaning strategy.

## Optional QC Table

This table is not necessary for the report figure, but it can help check how pruning affects skeleton pixels, endpoints, and branchpoint-like pixels.

In [ ]:
def skeleton_qc_table(cases: list[dict], skeleton_results: dict) -> pd.DataFrame:
    rows = []
    for strategy_name, by_case in skeleton_results.items():
        for case in cases:
            result = by_case[case["name"]]
            skeleton = result["skeleton"]
            endpoints = result["endpoints"]
            branchpoints = result["branchpoints"]
            rows.append(
                {
                    "strategy": strategy_name,
                    "case": case["name"],
                    "skeleton_pixels": int(np.count_nonzero(skeleton)),
                    "endpoint_pixels": int(np.count_nonzero(endpoints)),
                    "branchpoint_like_pixels": int(np.count_nonzero(branchpoints)),
                    "component_count": skeleton_component_count(skeleton),
                    "pre_min_area": result["pre_min_area"],
                    "skeleton_min_size": result["skeleton_min_size"],
                    "spur_prune_length": result["spur_prune_length"],
                }
            )
    return pd.DataFrame(rows)


skeleton_qc = skeleton_qc_table(cases, skeleton_results)
skeleton_qc